In [20]:
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [26]:
import re

def build_payload(query, agent):
    query = query.lower()
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    for tool in agent["tools"]:
        name = tool["name"]

        if name == "add" and "add" in query:
            return {
                "tool": "add",
                "args": {
                    "a": numbers[0],
                    "b": numbers[1]
                }
            }

        if name == "calculate_interest" and "interest" in query:
            return {
                "tool": "calculate_interest",
                "args": {
                    "principal": numbers[0],
                    "rate": numbers[1],
                    "time": numbers[2]
                }
            }

    return None

In [33]:
def negotiate(query, registry):  ##Broadcast Query
   
    responses = []

    for agent in registry:
        print(f"\n🔎 Trying agent: {agent['agent']}")

        payload = build_payload(query, agent)

        print("📦 Payload:", payload)

        if not payload:
            print("⚠️ Skipping (no matching tool)")
            continue

        try:
            res = requests.post(agent["endpoint"], json=payload).json()

            print("📨 Response:", res)

            responses.append({
                "agent": agent["agent"],
                "response": res
            })

        except Exception as e:
            print("❌ Error:", e)

    return responses

In [34]:
def evaluate(responses):  ##Evaluation Logic
    for r in responses:
        if r["response"].get("status") == "success":
            return r

    return {"error": "No valid response"}

In [35]:
def execute(query):  ## . Full Negotiation Flow
    print("\n🔍 QUERY:", query)

    registry = discover_capabilities()

    responses = negotiate(query, registry)

    print("\n📨 RESPONSES:")
    for r in responses:
        print(r)

    best = evaluate(responses)

    print("\n🏆 SELECTED:", best)

    return best

In [30]:
execute("add 10 and 20")


🔍 QUERY: add 10 and 20

🔎 Trying agent: math-agent
📦 Payload: {'tool': 'add', 'args': {'a': 10.0, 'b': 20.0}}
📨 Response: {'status': 'success', 'result': 30.0}

🔎 Trying agent: finance-agent
📦 Payload: None
⚠️ Skipping (no matching tool)

📨 RESPONSES:
{'agent': 'math-agent', 'response': {'status': 'success', 'result': 30.0}}

🏆 SELECTED: {'agent': 'math-agent', 'response': {'status': 'success', 'result': 30.0}}


{'agent': 'math-agent', 'response': {'status': 'success', 'result': 30.0}}

## Whats Happening 

Query → Send to multiple agents
        ↓
   Each agent responds
        ↓
Controller evaluates
        ↓
Best result selected

## Architecture   

Query
  ↓
Discovery
  ↓
Send to multiple agents (parallel)
  ↓
Collect responses
  ↓
Evaluation (A2A)
  ↓
Select best

## ⚙️ 4. Strategy
Query
  ↓
Discovery
  ↓
Send to multiple agents (parallel)
  ↓
Collect responses
  ↓
Evaluation (A2A)
  ↓
Select best

## 💥 9. What Just Happened
Step	Type
Broadcast	A2A
Multiple execution	MCP
Evaluation	A2A
## 🧠 10. Key Difference
Feature	Routing	Negotiation
Agents called	1	Many
Decision time	Before execution	After execution
Intelligence	Low	High